# 🛠️ Notebook 04: Feature Engineering

**Project:** Machine Learning-Based Prediction of Agricultural Residue Burning Activity Across Districts of Northwestern India

### Objective
Transform the integrated multi-source dataset into a structured feature matrix suitable for machine learning regression models.

### Prediction Task Definition
- **Target:** Predict next-day district-level agricultural fire counts (ires_day).
- **Leakage Prevention:** All features are strictly calculated using observations from prior dates (-1, t-2, \dots$). No future or same-day target data is used.
- **Target Integrity:** Missing target values represent unmonitored days and are not imputed as zero. Modeling is conducted on verified observation dates.


## 1. Import Libraries and Load Integrated Dataset

We import standard scientific computing libraries and load master_integrated_daily.csv.


In [1]:
import os
import pandas as pd
import numpy as np

# Robust path handling whether run from root or notebooks/
if os.path.exists('Data'):
    DATA_DIR = 'Data'
elif os.path.exists(os.path.join('..', 'Data')):
    DATA_DIR = os.path.join('..', 'Data')
else:
    DATA_DIR = 'data'

PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
INTEGRATED_DIR = os.path.join(PROCESSED_DIR, 'Integrated')
FEATURE_DIR = os.path.join(PROCESSED_DIR, 'feature')
os.makedirs(FEATURE_DIR, exist_ok=True)

master_path = os.path.join(INTEGRATED_DIR, 'master_integrated_daily.csv')
df = pd.read_csv(master_path, low_memory=False)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['district_standardized', 'date']).reset_index(drop=True)

print('Loaded master integrated dataset:')
print('Shape:', df.shape)
print('Date range:', df['date'].min().strftime('%Y-%m-%d'), 'to', df['date'].max().strftime('%Y-%m-%d'))
print('Districts:', df['district_standardized'].nunique())


Loaded master integrated dataset:
Shape: (47397, 34)
Date range: 2016-10-01 to 2026-10-27
Districts: 46


## 2. Satellite Detection Handling and Lag Features

Satellite thermal detections from FIRMS (MODIS, VIIRS, SNPP) are thermal anomalies detected from space.
When a district has no thermal detections on a date, we record zero detections. To predict next-day burning,
we compute 1-day, 2-day, and 3-day lagged detections and backward-looking rolling statistics.


In [2]:
# Fill missing satellite detection counts with 0 on non-detection days
sat_columns = ["modis_detections", "snpp_detections", "viirs_detections", "total_sensor_detections"]
for c in sat_columns:
    df[c] = df[c].fillna(0)
df["has_satellite_detection"] = (df["firms_matched"] == True).astype(int)

# Group by district to compute strictly lagged features
grouped = df.groupby("district_standardized")

# 1-day, 2-day, and 3-day satellite detection lags
df["sat_lag1"] = grouped["total_sensor_detections"].shift(1).fillna(0)
df["sat_lag2"] = grouped["total_sensor_detections"].shift(2).fillna(0)
df["sat_lag3"] = grouped["total_sensor_detections"].shift(3).fillna(0)

# 3-day and 7-day rolling average detections (shifted by 1 day to prevent leakage)
df["sat_roll_mean_3d"] = grouped["total_sensor_detections"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).fillna(0)

df["sat_roll_mean_7d"] = grouped["total_sensor_detections"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
).fillna(0)

df["sat_roll_max_7d"] = grouped["total_sensor_detections"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).max()
).fillna(0)

# Sensor-specific 1-day lags
df["sat_modis_lag1"] = grouped["modis_detections"].shift(1).fillna(0)
df["sat_viirs_lag1"] = grouped["viirs_detections"].shift(1).fillna(0)
df["sat_snpp_lag1"] = grouped["snpp_detections"].shift(1).fillna(0)

print("Satellite lag and rolling features computed successfully!")
print(df[["date", "district_standardized", "total_sensor_detections", "sat_lag1", "sat_roll_mean_3d"]].head(6))


Satellite lag and rolling features computed successfully!
        date district_standardized  ...  sat_lag1  sat_roll_mean_3d
0 2016-10-01                ambala  ...       0.0          0.000000
1 2016-10-02                ambala  ...       4.0          4.000000
2 2016-10-03                ambala  ...       0.0          2.000000
3 2016-10-04                ambala  ...       0.0          1.333333
4 2016-10-05                ambala  ...       1.0          0.333333
5 2016-10-06                ambala  ...       4.0          1.666667

[6 rows x 5 columns]


## 3. Historical Burning Activity (CREAMS Lag & Rolling Features)

We compute prior-day burning activity (creams_lag1) and 3-day and 7-day rolling means.
Because CREAMS observations have missing days, we add a binary indicator creams_lag1_is_missing
so models can distinguish a known zero burning day from an unmonitored day.


In [3]:
# Prior day CREAMS target lag
df["creams_lag1"] = grouped["fires_day"].shift(1)
df["creams_lag1_is_missing"] = df["creams_lag1"].isna().astype(int)
df["creams_lag1"] = df["creams_lag1"].fillna(0)

# 3-day and 7-day rolling average burning (shifted strictly by 1 day)
df["creams_roll_mean_3d"] = grouped["fires_day"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).fillna(0)

df["creams_roll_mean_7d"] = grouped["fires_day"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
).fillna(0)

print("CREAMS historical lag and rolling features computed successfully!")
print(df[["date", "district_standardized", "fires_day", "creams_lag1", "creams_lag1_is_missing"]].head(6))


CREAMS historical lag and rolling features computed successfully!
        date district_standardized  ...  creams_lag1  creams_lag1_is_missing
0 2016-10-01                ambala  ...          0.0                       1
1 2016-10-02                ambala  ...          0.0                       1
2 2016-10-03                ambala  ...          0.0                       1
3 2016-10-04                ambala  ...          0.0                       1
4 2016-10-05                ambala  ...          0.0                       1
5 2016-10-06                ambala  ...          0.0                       1

[6 rows x 5 columns]


## 4. Calendar and Seasonality Features

Stubble burning in Northwestern India follows a strong seasonal cycle closely tied to the Kharif paddy harvest,
typically peaking between mid-October and mid-November.
We extract calendar components, day-of-season counters, and cyclical sine/cosine transformations.


In [4]:
# Calendar components
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["day_of_year"] = df["date"].dt.dayofyear

# Days elapsed since September 15 (start of post-monsoon residue burning period)
season_start = pd.to_datetime(df["year"].astype(str) + "-09-15")
df["day_of_season"] = (df["date"] - season_start).dt.days

# Cyclical encoding of seasonal progression
df["sin_day_of_season"] = np.sin(2 * np.pi * df["day_of_season"] / 90.0)
df["cos_day_of_season"] = np.cos(2 * np.pi * df["day_of_season"] / 90.0)

# Peak harvesting/burning window indicator (October 15 to November 20)
df["is_peak_window"] = (
    ((df["month"] == 10) & (df["day"] >= 15)) |
    ((df["month"] == 11) & (df["day"] <= 20))
).astype(int)

print("Seasonality features generated:")
print(df[["date", "month", "day", "day_of_season", "is_peak_window"]].head(6))


Seasonality features generated:
        date  month  day  day_of_season  is_peak_window
0 2016-10-01     10    1             16               0
1 2016-10-02     10    2             17               0
2 2016-10-03     10    3             18               0
3 2016-10-04     10    4             19               0
4 2016-10-05     10    5             20               0
5 2016-10-06     10    6             21               0


## 5. Agricultural Capacity and Spatial Features

Rice cultivation acreage directly determines the quantity of paddy stubble generated.
We include district Kharif rice area, rice production, total agricultural area, and geographic coordinates.


In [5]:
# Spatial imputation for valid newer districts missing boundary polygons
df["centroid_lat"] = df.groupby("state_creams")["centroid_lat"].transform(lambda x: x.fillna(x.median()))
df["centroid_lon"] = df.groupby("state_creams")["centroid_lon"].transform(lambda x: x.fillna(x.median()))
df["boundary_area_sqkm"] = df.groupby("state_creams")["boundary_area_sqkm"].transform(lambda x: x.fillna(x.median()))

# Agricultural intensity features
df["rice_intensity"] = df["kharif_area_ha"] / (df["boundary_area_sqkm"] * 100.0 + 1.0)
df["rice_prod_per_ha"] = df["kharif_prod_t"] / (df["kharif_area_ha"] + 1.0)

# State indicator (Punjab = 1, Haryana = 0)
df["is_punjab"] = (df["state_creams"] == "Punjab").astype(int)

print("Agricultural and spatial features computed!")
print(df[["district_standardized", "state_creams", "kharif_area_ha", "rice_intensity", "is_punjab"]].head())


Agricultural and spatial features computed!
  district_standardized state_creams  kharif_area_ha  rice_intensity  is_punjab
0                ambala      Haryana         83000.0        0.422181          0
1                ambala      Haryana         83000.0        0.422181          0
2                ambala      Haryana         83000.0        0.422181          0
3                ambala      Haryana         83000.0        0.422181          0
4                ambala      Haryana         83000.0        0.422181          0


## 6. Export Modeling Dataset and Quality Summary

We export the complete feature matrix and create the modeling subset where verified target values (ires_day) exist.


In [6]:
# 1. Save full daily feature dataset
full_feat_path = os.path.join(FEATURE_DIR, "full_daily_features.csv")
df.to_csv(full_feat_path, index=False)
print("Saved full daily features to:", full_feat_path)

# 2. Filter to verified target observation records for ML modeling
modeling_df = df[df["fires_day"].notna()].copy()
model_feat_path = os.path.join(FEATURE_DIR, "modeling_dataset_daily.csv")
modeling_df.to_csv(model_feat_path, index=False)

print("=" * 60)
print("FEATURE ENGINEERING COMPLETED SUCCESSFULLY!")
print("=" * 60)
print("Full dataset shape:", df.shape)
print("Modeling dataset shape:", modeling_df.shape)
print("Target fires_day count:", len(modeling_df))
print("Target fires_day mean:", round(modeling_df["fires_day"].mean(), 2))
print("Target fires_day max:", modeling_df["fires_day"].max())
print("Seasons present:", sorted(modeling_df["season_year"].unique()))


Saved full daily features to: Data\processed_data\feature\full_daily_features.csv
FEATURE ENGINEERING COMPLETED SUCCESSFULLY!
Full dataset shape: (47397, 61)
Modeling dataset shape: (14392, 61)
Target fires_day count: 14392
Target fires_day mean: 30.69
Target fires_day max: 893.0
Seasons present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
